In [1]:
import warnings
warnings.filterwarnings('always')
warnings.filterwarnings('ignore')

import pandas as pd
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))
pd.set_option('display.max_columns', None)
import numpy as np

#statistics
import scipy.stats as st

#visulaization
import geopandas as gpd
import matplotlib.pyplot as plt

#mlxtend
from matplotlib.colors import LinearSegmentedColormap
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules, fpmax



# Data Read

In [32]:
df = pd.read_csv('./INPUT/full_merge_test_score.csv')

In [47]:
colist = [x for x in df.columns if 'road' in x] + [x for x in df.columns if 'func' in x] + ['교통사고', '도로', '동영상', '마음가짐', '면허취득', '법규준수', '안전운전','안전표지', '인명', '자전거, 친환경', '점검, 관리', '특별상황']

colist = [x for x in colist if 'RESPOND' not in x and x not in ['func_'+x for x in ['HEADLIGHT_PNT', 'TURN_SIG_LMP_PNT', 'WIPER_PNT', 'GEAR_CHG_PNT', 'OBEY_CAR_ROAD_PNT', 'SUDDEN_STOP_PNT']]]


In [48]:
df = df[colist]

In [49]:
df.columns

Index(['road_BEF_DECT_PNT', 'road_DRV_DECT_PNT', 'road_STR_DECT_PNT',
       'road_SPD_DECT_PNT', 'road_BRAKE_DECT_PNT', 'road_STER_DECT_PNT',
       'road_BODY_DECT_PNT', 'road_NEVI_DECT_PNT', 'road_CHG_DECT_PNT',
       'road_TURN_DECT_PNT', 'road_PARK_DECT_PNT', 'road_ETC_DECT_PNT',
       'func_START_PNT', 'func_END_PNT', 'func_BELT_PNT', 'func_CROSS_PNT',
       'func_CLIMB_PNT', 'func_SIGNAL_PNT', 'func_RPMOVER_PNT',
       'func_SPEEDOVER_PNT', 'func_STARTING_PNT', 'func_ZCOUR_PNT',
       'func_SCOUR_PNT', 'func_TCOUR_PNT', 'func_PCOUR_PNT', 'func_SUDDEN_PNT',
       'func_GEAR_PNT', 'func_TIME_PNT', '교통사고', '도로', '동영상', '마음가짐', '면허취득',
       '법규준수', '안전운전', '안전표지', '인명', '자전거, 친환경', '점검, 관리', '특별상황'],
      dtype='object')

In [50]:
df.columns = [
 '도로주행_출발전확인감점', '도로주행_운전자세감점', '도로주행_출발감점', 
    '도로주행_가속및속도유지감점', '도로주행_제동감점', '도로주행_조향감점', 
    '도로주행_자체감각감점', '도로주행_통행구분감점', '도로주행_진로변경감점',
 '도로주행_직진및좌우회전감점', '도로주행_주차방법감점', '도로주행_기타감점',

'장내기능_출발감점', '장내기능_종료감점', '장내기능_벨트감점', '장내기능_횡단감점', 
    '장내기능_경사감점',  '장내기능_신호감점',  '장내기능_RPM감점', 
    '장내기능_속도위반감점', '장내기능_시동감점', '장내기능_Z-COURSE감점', 
    '장내기능_S-COURSE감점', '장내기능_T-COURSE감점', '장내기능_P-COURSE감점', '장내기능_돌발감점', 
    '장내기능_기어감점', '장내기능_시간초과감점', '교통사고', '도로', '동영상', '마음가짐', '면허취득', '법규준수', 
 '안전운전','안전표지', '인명', '자전거, 친환경', '점검, 관리', '특별상황']

In [65]:
df.to_csv('./INPUT/full_merge_only_score.csv',index=False)

In [56]:
df.columns

Index(['도로주행_출발전확인감점', '도로주행_운전자세감점', '도로주행_출발감점', '도로주행_가속및속도유지감점',
       '도로주행_제동감점', '도로주행_조향감점', '도로주행_자체감각감점', '도로주행_통행구분감점', '도로주행_진로변경감점',
       '도로주행_직진및좌우회전감점', '도로주행_주차방법감점', '도로주행_기타감점', '장내기능_출발감점', '장내기능_종료감점',
       '장내기능_벨트감점', '장내기능_횡단감점', '장내기능_경사감점', '장내기능_신호감점', '장내기능_RPM감점',
       '장내기능_속도위반감점', '장내기능_시동감점', '장내기능_Z-COURSE감점', '장내기능_S-COURSE감점',
       '장내기능_T-COURSE감점', '장내기능_P-COURSE감점', '장내기능_돌발감점', '장내기능_기어감점',
       '장내기능_시간초과감점', '교통사고', '도로', '동영상', '마음가짐', '면허취득', '법규준수', '안전운전',
       '안전표지', '인명', '자전거, 친환경', '점검, 관리', '특별상황'],
      dtype='object')

In [59]:
['PC학과_' + x for x in ['교통사고', '도로', '동영상', '마음가짐', '면허취득', '법규준수', '안전운전','안전표지', '인명', '자전거, 친환경', '점검, 관리', '특별상황']]

['PC학과_교통사고',
 'PC학과_도로',
 'PC학과_동영상',
 'PC학과_마음가짐',
 'PC학과_면허취득',
 'PC학과_법규준수',
 'PC학과_안전운전',
 'PC학과_안전표지',
 'PC학과_인명',
 'PC학과_자전거, 친환경',
 'PC학과_점검, 관리',
 'PC학과_특별상황']

In [60]:
df.columns = ['도로주행_출발전확인감점', '도로주행_운전자세감점', '도로주행_출발감점', '도로주행_가속및속도유지감점',
       '도로주행_제동감점', '도로주행_조향감점', '도로주행_자체감각감점', '도로주행_통행구분감점', '도로주행_진로변경감점',
       '도로주행_직진및좌우회전감점', '도로주행_주차방법감점', '도로주행_기타감점', '장내기능_출발감점', '장내기능_종료감점',
       '장내기능_벨트감점', '장내기능_횡단감점', '장내기능_경사감점', '장내기능_신호감점', '장내기능_RPM감점',
       '장내기능_속도위반감점', '장내기능_시동감점', '장내기능_Z-COURSE감점', '장내기능_S-COURSE감점',
       '장내기능_T-COURSE감점', '장내기능_P-COURSE감점', '장내기능_돌발감점', '장내기능_기어감점',
       '장내기능_시간초과감점', 'PC학과_교통사고', 'PC학과_도로', 'PC학과_동영상', 'PC학과_마음가짐', 'PC학과_면허취득', 'PC학과_법규준수',
 'PC학과_안전운전', 'PC학과_안전표지', 'PC학과_인명', 'PC학과_자전거, 친환경', 'PC학과_점검, 관리', 'PC학과_특별상황']

In [63]:
df = df[
[ 'PC학과_교통사고', 'PC학과_도로', 'PC학과_동영상', 'PC학과_마음가짐',
       'PC학과_면허취득', 'PC학과_법규준수', 'PC학과_안전운전', 'PC학과_안전표지', 'PC학과_인명',
       'PC학과_자전거, 친환경', 'PC학과_점검, 관리', 'PC학과_특별상황',
 '장내기능_출발감점', '장내기능_종료감점',
       '장내기능_벨트감점', '장내기능_횡단감점', '장내기능_경사감점', '장내기능_신호감점', '장내기능_RPM감점',
       '장내기능_속도위반감점', '장내기능_시동감점', '장내기능_Z-COURSE감점', '장내기능_S-COURSE감점',
       '장내기능_T-COURSE감점', '장내기능_P-COURSE감점', '장내기능_돌발감점', '장내기능_기어감점',
       '장내기능_시간초과감점',
 '도로주행_출발전확인감점', '도로주행_운전자세감점', '도로주행_출발감점', '도로주행_가속및속도유지감점',
       '도로주행_제동감점', '도로주행_조향감점', '도로주행_자체감각감점', '도로주행_통행구분감점', '도로주행_진로변경감점',
       '도로주행_직진및좌우회전감점', '도로주행_주차방법감점', '도로주행_기타감점', ]
]

In [64]:
df

,PC학과_교통사고,PC학과_도로,PC학과_동영상,PC학과_마음가짐,PC학과_면허취득,PC학과_법규준수,PC학과_안전운전,PC학과_안전표지,PC학과_인명,"PC학과_자전거, 친환경","PC학과_점검, 관리",PC학과_특별상황,장내기능_출발감점,장내기능_종료감점,장내기능_벨트감점,장내기능_횡단감점,장내기능_경사감점,장내기능_신호감점,장내기능_RPM감점,장내기능_속도위반감점,장내기능_시동감점,장내기능_Z-COURSE감점,장내기능_S-COURSE감점,장내기능_T-COURSE감점,장내기능_P-COURSE감점,장내기능_돌발감점,장내기능_기어감점,장내기능_시간초과감점,도로주행_출발전확인감점,도로주행_운전자세감점,도로주행_출발감점,도로주행_가속및속도유지감점,도로주행_제동감점,도로주행_조향감점,도로주행_자체감각감점,도로주행_통행구분감점,도로주행_진로변경감점,도로주행_직진및좌우회전감점,도로주행_주차방법감점,도로주행_기타감점
0,0.0,0.22,1.0,0.50,0.0,0.50,0.57,0.4,0.0,0.00,0.50,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,6.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0,0,0,0,7,0,0,0
1,1.0,0.11,0.0,0.33,1.0,0.00,0.43,0.0,0.0,0.00,0.50,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0,0,0,0,7,14,0,0
2,0.0,0.44,1.0,0.50,0.5,0.25,0.43,0.4,0.0,0.00,0.50,0.5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,10,0,0,0,0,0,0,0,0,0,0
3,0.0,0.00,0.0,0.33,0.0,0.50,0.29,0.0,0.5,0.25,0.50,0.0,0.0,5.0,0.0,0.0,0.0,0.0,0.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,10.0,0.0,0,0,0,0,5,0,0,0,0,14,0,0
4,0.0,0.00,0.0,0.00,0.0,0.75,0.29,0.2,0.0,0.00,0.00,0.0,5.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0,0,0,0,14,7,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
132171,1.0,0.22,0.0,0.00,0.0,0.40,0.29,0.8,0.0,0.00,0.33,0.0,0.0,0.0,0.0,0.0,10.0,0.0,0.0,0.0,0.0,0.0,0.0,10.0,0.0,0.0,0.0,0.0,0,0,0,0,0,0,0,0,7,0,0,0
132172,0.0,0.00,0.0,1.00,0.0,1.00,0.00,0.2,0.0,0.00,0.50,0.0,0.0,5.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,5,0,0,0,0,0,0,0,0,0,0
132173,0.0,0.22,0.0,0.00,0.0,0.60,0.00,0.0,0.0,0.33,0.50,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0,7,0,0,0,0,0,0
132174,0.0,0.33,0.0,0.00,1.0,0.20,0.71,0.6,0.5,0.00,1.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,15,0,0,7,0,0,0,7,0,0,0


In [128]:
func = pd.read_csv('../../../011.데이터/채점데이터/_SELECT_FROM_DISEXAM08_TB_RERRESTEXP_tr_inner_JOIN_DISEXAM08_TB__08_20231101.csv')
road = pd.read_csv('../../../011.데이터/채점데이터/TB_RERROADEXP_08_20231101.csv')

In [131]:
func.drop(columns = ['RESIDENT_NAME','RESIDENT_DATE'],inplace=True)

In [137]:
road.drop(columns = ['RESIDENT_DATE','RESIDENT_NO','RESIDENT_DATE','RES_PHONE_NO'],inplace=True)

In [139]:
func.head().to_csv('./use_data/sample_func.csv',index=False)
road.head().to_csv('./use_data/sample_road.csv',index=False)

# PreProcessing
- mlxtend 시 2차원 리스트 형태가 필요함


In [3]:
raw = pd.read_csv('../../../011.데이터/채점데이터/_SELECT_ui_U_NAME_ui_U_JUMIN_NO_uec_U_INFO_ID_uec_Q_ID_qq_L_CD_q_20231101.csv',nrows=200000)
info = pd.read_csv('../../../011.데이터/채점데이터/U_INFO_202311011052.csv')
cd = pd.read_csv('../../../011.데이터/채점데이터/QMS_UNITCODE_202311011107.csv')

In [4]:
raw.head()

,U_NAME,U_JUMIN_NO,U_INFO_ID,Q_ID,L_CD,M_CD,UNITCODE_NAME,Q_COUNT,CORR_ANSWER,U_ANSWER,CORRECT_YN,Q_LEVEL,U_EXAM_OPEN,U_TYPE,U_CUTLINE
0,홍정은,9309012******,35021,781,19,26,도로,28,"1,5","1,5",1,0,2022-01-03 09:00:00.000,2,60
1,홍정은,9309012******,35021,801,19,27,도로,29,"1,3","1,3",1,0,2022-01-03 09:00:00.000,2,60
2,홍정은,9309012******,35021,816,19,28,도로,30,"2,3","2,3",1,0,2022-01-03 09:00:00.000,2,60
3,홍정은,9309012******,35021,826,19,29,도로,31,"3,5","5,3",1,0,2022-01-03 09:00:00.000,2,60
4,홍정은,9309012******,35021,838,19,30,도로,32,"2,4","2,3",0,0,2022-01-03 09:00:00.000,2,60


In [9]:
raw['IDX'] = raw['U_NAME']+'_'+raw['U_JUMIN_NO']

In [17]:
info[(info['U_NAME'] =='배기찬')&(info['U_JUMIN_NO'].str.contains('000820'))]

,ID,EXM_CD,U_RECEIVE_NO,U_EXAM_NO,U_EXAM_OPEN,U_CNT,U_NAME,U_TYPE,U_LEVEL,U_EXAM_TYPE,U_CUTLINE,U_EXAMTIME,U_TIMELEFT,U_ABSENT,U_CORRECTCNT,U_EXAM_COMMENT,U_SCORE,U_JUMIN_NO,U_ERR,U_LANG,U_PASSFAIL,U_SEAT_NO,U_SEAT_IP,U_R_QS_TYPE,COMPULSORY,USER_ID,TEST_MODE,HANDY_OPT,REM_GNUS_CODE,GENDER,AGE,Q_GUBUN,Q_ITEM,PAPER_ID,U_TIMETAKEN
9,34995,1000,10210004589,22,2022-01-03 09:00:00.000,624948,배기찬,2,보통,A-A-10945,60,40,10,1,24.0,16_01_4_2_X;49_02_3_1_X;75_03_2_2_O;132_04_2_2...,59.0,0008203******,NaN,1,2.0,65,192.168.10.165,5,NaN,설선미,N,0,0,1,21,A,A,10945,30
1460,37139,1000,10210004589,52,2022-01-10 09:00:00.000,627060,배기찬,2,보통,A-A-10931,60,40,6,1,21.0,34_01_3_3_O;47_02_4_4_O;65_03_3_2_X;84_04_4_1_...,54.0,0008203******,NaN,1,2.0,33,192.168.10.133,2,NaN,설선미,N,0,0,1,21,A,A,10931,34
2163,36492,1000,10210004589,30,2022-01-07 09:00:00.000,626453,배기찬,2,보통,A-A-10936,60,40,4,1,21.0,30_01_1_3_X;56_02_2_3_X;70_03_1_1_O;106_04_3_3...,54.0,0008203******,NaN,1,2.0,5,192.168.10.105,2,NaN,설선미,N,0,0,1,21,A,A,10936,36
7229,42566,1000,10220007406,79,2022-01-21 09:00:00.000,632308,배기찬,2,보통,A-A-10931,60,40,14,1,20.0,34_01_3_3_O;47_02_4_4_O;65_03_3_2_X;84_04_4_1_...,52.0,0008203******,NaN,1,2.0,14,192.168.10.114,5,NaN,김주현,N,0,0,1,21,A,A,10931,26
9684,44031,1000,10220007406,25,2022-01-26 09:00:00.000,634002,배기찬,2,보통,A-A-10929,60,40,16,1,24.0,36_01_2_2_O;38_02_3_3_O;66_03_1_4_X;121_04_1_1...,60.0,0008203******,NaN,1,1.0,61,192.168.10.161,8,NaN,설선미,N,0,0,1,21,A,A,10929,24


In [20]:
raw[raw['U_INFO_ID'] == 42566]

,U_NAME,U_JUMIN_NO,U_INFO_ID,Q_ID,L_CD,M_CD,UNITCODE_NAME,Q_COUNT,CORR_ANSWER,U_ANSWER,CORRECT_YN,Q_LEVEL,U_EXAM_OPEN,U_TYPE,U_CUTLINE,IDX


# Chi Squared

# Cramer's V

# mlxtend
- Apriori 알고리즘
    - 최소 지지도, 최소 신뢰도를 갖는 연관규칙을 찾는 방법

## Transform
- 2차원 리스트 형태로 변환시켜야 함
 - 전처리를 수행할 때 

# Road Test Course

In [24]:
import shapely

In [2]:
gdf = gpd.read_file('../../../011.데이터/08.LICENSE_ETC/test_cos_line.shp')

In [26]:
flip_gdf = gdf.copy()

In [27]:
flip_gdf['geometry'] = gpd.GeoSeries(gdf['geometry']).map(lambda polygon: shapely.ops.transform(lambda x, y: (y, x), polygon))

In [30]:
flip_gdf.to_csv('./use_data/gis/FLIP_COS_LINE.csv',index=False)

In [31]:
flip_gdf.crs

<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich